In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert
import os
import time

# ==============================================================================
# 0. DOSSIER WINDOWS DU PROJET + TIMER
# ==============================================================================

start_time = time.time()

project = "ERT_Project_Windows"
base_dir = os.path.abspath(project)

fig_dir = os.path.join(base_dir, "Figures")
data_dir = os.path.join(base_dir, "Data")
log_dir = os.path.join(base_dir, "Logs")

for d in [fig_dir, data_dir, log_dir]:
    os.makedirs(d, exist_ok=True)

log_file = os.path.join(log_dir, "run_log.txt")

geometry_file = os.path.join(fig_dir, "01_geometry.png")
true_model_file = os.path.join(fig_dir, "02_true_resistivity_model.png")
mesh_file = os.path.join(fig_dir, "03_mesh.png")
pseudosection_file = os.path.join(fig_dir, "04_pseudosection.png")
inversion_file = os.path.join(fig_dir, "05_inversion.png")

topo_file = os.path.join(data_dir, "topo.csv")
mesh_file_bms = os.path.join(data_dir, "mesh.bms")
ert_data_file = os.path.join(data_dir, "ert_data.dat")
model_npy_file = os.path.join(data_dir, "model.npy")
runtime_file = os.path.join(log_dir, "runtime.txt")

def log(txt):
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(txt + "\n")
    print(txt)

def save_figure(fig, path, title=None):
    if title is not None:
        fig.suptitle(title)
    fig.canvas.draw()
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    log(f"Figure sauvegardée : {path}")

log("=== RUN START ===")
log(f"Dossier projet : {base_dir}")
log(f"Géométrie : {geometry_file}")
log(f"Modèle vrai : {true_model_file}")
log(f"Maillage : {mesh_file}")
log(f"Pseudosection : {pseudosection_file}")
log(f"Inversion : {inversion_file}")

# ==============================================================================
# 1. GÉOMÉTRIE
# ==============================================================================

n_electrodes = 36
spacing = 5
x_max = (n_electrodes - 1) * spacing
z_bottom = 95.0

x_topo = np.linspace(0, x_max, n_electrodes)
y_topo = 162.0 - 1.5 * np.sin(np.pi * x_topo / x_max)
points_topo = list(zip(x_topo, y_topo))

np.savetxt(
    topo_file,
    np.column_stack([x_topo, y_topo]),
    delimiter=",",
    header="X,Y",
    comments="")
log(f"Topographie sauvegardée : {topo_file}")

poly_points = points_topo + [[x_max, z_bottom], [0, z_bottom]]
world = mt.createPolygon(poly_points, isClosed=True, marker=1)

for p in points_topo:
    world.createNode(p)

points_socle = [
    [0, z_bottom],
    [40.0, 112.0],
    [85.0, 142.0],
    [125.0, 115.0],
    [x_max, 110.0],
    [x_max, z_bottom]]

socle = mt.createPolygon(points_socle, isClosed=True, marker=2)

poche = mt.createCircle(pos=[90.0, 155.0], radius=[80,8], marker=3, area=1)

geometry = world + socle + poche

fig, ax = plt.subplots(figsize=(10, 5))
pg.show(geometry, ax=ax)
ax.set_title("Geometry model")
save_figure(fig, geometry_file)

# ==============================================================================
# 2. MAILLAGE
# ==============================================================================

mesh = mt.createMesh(geometry, quality=34, area=2.0, smooth=True)
mesh.save(mesh_file_bms)
log(f"Maillage sauvegardé : {mesh_file_bms}")

fig, ax = plt.subplots(figsize=(10, 5))
pg.show(mesh, ax=ax, showMesh=True)
ax.set_title("Mesh")
save_figure(fig, mesh_file)

# ==============================================================================
# 3. MODÈLE DE RÉSISTIVITÉ VRAI
# ==============================================================================
# Construction d'un modèle cellulaire explicite pour l'affichage
true_res = np.ones(mesh.cellCount()) * 50.0

for cell in mesh.cells():
    m = cell.marker()
    if m == 2:
        true_res[cell.id()] = 150.0
    elif m == 3:
        true_res[cell.id()] = 3.0
    elif m == 1:
        true_res[cell.id()] = 50.0
    else:
        true_res[cell.id()] = 50.0

fig, ax = plt.subplots(figsize=(10, 5))
pg.show(
    mesh,
    data=true_res,
    ax=ax,
    showMesh=True,
    cMap="jet",
    logScale=True)
ax.set_title("True resistivity model")
save_figure(fig, true_model_file)

# ==============================================================================
# 4. ACQUISITION
# ==============================================================================
scheme = ert.createData(elecs=points_topo, schemeName='wa')

# ==============================================================================
# 5. SIMULATION
# ==============================================================================
rhomap = [
    [0, 10.0],
    [1, 50.0],
    [2, 150.0],
    [3, 2.0]]

data = ert.simulate(
    mesh,
    scheme=scheme,
    res=rhomap,
    noiseLevel=0.01,
    noiseAbs=1e-6,
    seed=2025)

data.remove(data['rhoa'] <= 0)
data.save(ert_data_file)
log(f"Données ERT sauvegardées : {ert_data_file}")

fig, ax = plt.subplots(figsize=(10, 5))
ert.show(data, ax=ax)
ax.set_title("Data pseudosection")
save_figure(fig, pseudosection_file)
# ==============================================================================
# 6. INVERSION
# ==============================================================================

mgr = ert.ERTManager(data, sr=True, verbose=True)

model = mgr.invert(
    lam=20,
    paraMaxCellSize=1.5,
    maxIter=10,
    zWeight=0.5,
    robustData=True,
    growthRate=1.1)

rms_final = mgr.inv.relrms()
log(f"RMS final = {rms_final:.2f}%")

np.save(model_npy_file, model)
log(f"Modèle inversé sauvegardé : {model_npy_file}")

fig, ax = plt.subplots(figsize=(12, 6))
mgr.showResult(
    ax=ax,
    cMin=3,
    cMax=150,
    cMap="jet",
    logScale=True)
ax.set_title(f"Inversion RMS = {rms_final:.2f}%")
save_figure(fig, inversion_file)

# ==============================================================================
# 7. LOG + TEMPS D'EXÉCUTION
# ==============================================================================

end_time = time.time()
duration_min = (end_time - start_time) / 60.0

with open(runtime_file, "w", encoding="utf-8") as f:
    f.write(f"Total runtime: {duration_min:.2f} minutes\n")
    f.write(f"Project folder: {base_dir}\n")
    f.write(f"Geometry file: {geometry_file}\n")
    f.write(f"True model file: {true_model_file}\n")
    f.write(f"Mesh file: {mesh_file}\n")
    f.write(f"Pseudosection file: {pseudosection_file}\n")
    f.write(f"Inversion file: {inversion_file}\n")

log(f"Execution time = {duration_min:.2f} minutes")
log(f"Runtime log saved : {runtime_file}")
log("=== RUN END ===")

print("\n✔ Projet terminé avec sauvegarde complète")
print("Dossier principal :", base_dir)
print("Géométrie :", geometry_file)
print("Modèle vrai :", true_model_file)
print("Maillage :", mesh_file)
print("Pseudosection :", pseudosection_file)
print("Inversion :", inversion_file)